# DC3

## Radix sort
Sort the numbers according to their digits in increasing order (from least to most significant digits)

In [718]:
from tqdm.notebook import tqdm
import copy
def digit_sort(array, exp):
    # initialize count array as 0
    count = [0 for _ in range(10)]
    sorted_array = []
    digit = []
    for i in range(len(array)):
        # initialize sorted_array
        sorted_array.append([0,0])
        # get current digit for each number 
        digit.append((array[i][0]//exp)%10)
 
    # Store count of occurrences in count
    for i in range(len(array)):
        index = digit[i]
        count[index] += 1
 
    # Change count[i] so that count[i] now contains actual position of this digit in sorted array
    for i in range(1, 10):
        count[i] += count[i - 1]
    # sort array
    for i in range(len(array)-1, -1, -1):
        index = digit[i]
        sorted_array[count[index] - 1][:] = array[i][:]
        #array_index[i] = count[index] - 1
        count[index] -= 1
    # update array to sorted_array
    i = 0
    for i in range(0, len(array)):
        array[i][:] = sorted_array[i][:]

def get_order(ordered_array):
    order = [1]
    prev_nb = ordered_array[0]
    repeated_char = False
    for i in range(1,len(ordered_array)):
        if ordered_array[i] > prev_nb:
            order.append(order[-1] + 1)
            prev_nb = ordered_array[i]
        else:
            order.append(order[-1])
            repeated_char = True
    return(order,repeated_char)


def get_ranks(index_ordered):
    ranks = {}
    for i,a in enumerate(index_ordered):
        ranks[a] = i+1
    return ranks

def radix_sort(array,max_=None,indexed=False): 
    # Add index to array
    if not indexed:
        array_index = [[a,i] for i,a in enumerate(array)]
    # Find the maximum number to know number of digits
        if max_ is None:
            max_ = max(array)
    else:
        array_index = array
        max_ = array[0][0]
        for a in array:
            if max_< a[0]:
                max_ = a[0]
    # Sort for every digit. Note that instead. Here exp is 10^i, where i is current digit number
    exp = 1
    while max_ / exp >= 1:
        digit_sort(array_index, exp)
        exp *= 10    
    array_sorted, index = tuple(zip(*array_index))
    order,repeated_char = get_order(array_sorted)
    return array_sorted, index, order, repeated_char


In [719]:
import numpy as np
np.random.seed(0)
arr = list(np.random.randint(0,100,10))
print(arr)
arr,index,order,repeated_char  = radix_sort(arr)
print(arr)
print(index)
print(order)
print(repeated_char)

[44, 47, 64, 67, 67, 9, 83, 21, 36, 87]
(9, 21, 36, 44, 47, 64, 67, 67, 83, 87)
(5, 7, 8, 0, 1, 2, 3, 4, 6, 9)
[1, 2, 3, 4, 5, 6, 7, 7, 8, 9]
True


# DC3

In [956]:
def string_2_numbers(string, alphabet=None, return_alphabet=False):
    if alphabet is None:
        alphabet = {char: i+1 for i,char in enumerate(sorted(list(set(string))))}
        alphabet_inverse = {alphabet[char]: char for char in alphabet}
    array = []
    for s in string:
        array.append(alphabet[s])
    if return_alphabet:
        return(array,alphabet,alphabet_inverse)
    else:
        return(array)

def append_sentinels(array,sentinel=0):
    return array+[sentinel,sentinel,sentinel]

def R_12(array,to_single_number=True,indexed=False):
    array_copy = array
    P_12 = list(range(1,len(array_copy)-2,3)) + list(range(2,len(array_copy)-2,3))
    if to_single_number:
        R_12 = [array_copy[p]*10**8+array_copy[p+1]*10**4+array_copy[p+2] for p in P_12]
    else:
        R_12 = [(array_copy[p],array_copy[p+1], array_copy[p+2]) for p in P_12]
    if indexed:
        return P_12,[[R_12[i], P_12[i]] for i in range(len(P_12))]
    return P_12,R_12

def R_0(array,ranks_R12,indexed=False):
    array_copy = array
    P_0 = list(range(0,len(array_copy)-3,3))
    R_0 = [array[p]*10**10+ranks_R12[p+1] for p in P_0]
    if indexed:
        return P_0,[[R_0[i], P_0[i]] for i in range(len(P_0))]
    return P_0,R_0

def resume_lower_order(index_12, order_12, p_12):
    mapper = {index_12[i] : order_12[i] for i in range(len(order_12))}
    return [mapper[p] for p in p_12]

def merge_indexes_R0_R12(index_R0, index_R12, ranks_R12, array):
    index = []
    i = 0
    j = 0
    while i<len(index_R0) and j<len(index_R12):
        a = index_R0[i]
        b = index_R12[j]
        if array[a] < array[b]:
            index.append(a)
            i += 1
        elif array[b] < array[a]:
            index.append(b)
            j += 1
        elif b%3 == 1:
            if ranks_R12[a+1] < ranks_R12[b+1]:
                index.append(a)
                i += 1
            elif ranks_R12[b+1] < ranks_R12[a+1]:
                index.append(b)
                j += 1
            else:
                print("error")
        elif b%3 == 2:
            if array[a+1] < array[b+1]:
                index.append(a)
                i += 1
            elif array[b+1] < array[a+1]:
                index.append(b)
                j += 1
            elif array[b+1] == array[a+1]:
                if ranks_R12[a+2] < ranks_R12[b+2]:
                    index.append(a)
                    i += 1
                elif ranks_R12[b+2] < ranks_R12[a+2]:
                    index.append(b)
                    j += 1
                else:
                    print("error")
    if i < len(index_R0):
        index += index_R0[i:]
    elif j < len(index_R12):
        index += index_R12[j:]
    return index

def resume_higher_order(index_order_n, p_order_n):
    print(len(p_order_n),len(index_order_n))
    return [p_order_n[i] for i in index_order_n]

def print_suffix_array(array,suffix_array):
    print("--------------")
    for i in suffix_array:
        print(array[i:],i)
    print("--------------")

def DC3(T):
    T = append_sentinels(T)
    p_12,r12_indexed = R_12(T,indexed=True)    
    r12_sorted, index_12, order_12, repeated_char_12  = radix_sort(copy.deepcopy(r12_indexed),indexed=True)
    if repeated_char_12:
        T_next = resume_lower_order(index_12, order_12, p_12)
        index_order_n = DC3(T_next)
        index_12_ = resume_higher_order(index_order_n, p_12)
        index_12 = index_12_
    ranks_12 = get_ranks(index_12)
    p_0,r0_indexed = R_0(T,ranks_12,indexed=True)
    r0_sorted, index_0, order_0, repeated_char_0  = radix_sort(copy.deepcopy(r0_indexed),indexed=True)
    index_012 = merge_indexes_R0_R12(index_0, index_12, ranks_12, T)
    if index_012[0] >= len(T)-3:
        index_012 = index_012[1:]
    return index_012
        
def BWT(T):
    suffix_array = DC3(T)
    bwt = [T[i-1] for i in suffix_array]
    return bwt

In [957]:
string = "abcabcacab"
Tp = string_2_numbers(string)
suffix_array = DC3(Tp)
for i in suffix_array:
    print(string[i:],i)


7 7
ab 8
abcabcacab 0
abcacab 3
acab 6
b 9
bcabcacab 1
bcacab 4
cab 7
cabcacab 2
cacab 5


In [958]:
string = "atcaaagattattatatatattagtttttttttgtttt"
Tp = string_2_numbers(string)
suffix_array = DC3(Tp)
for i in suffix_array:
    print(string[i:],i)

26 26
aaagattattatatatattagtttttttttgtttt 3
aagattattatatatattagtttttttttgtttt 4
agattattatatatattagtttttttttgtttt 5
agtttttttttgtttt 22
atatatattagtttttttttgtttt 13
atatattagtttttttttgtttt 15
atattagtttttttttgtttt 17
atcaaagattattatatatattagtttttttttgtttt 0
attagtttttttttgtttt 19
attatatatattagtttttttttgtttt 10
attattatatatattagtttttttttgtttt 7
caaagattattatatatattagtttttttttgtttt 2
gattattatatatattagtttttttttgtttt 6
gtttt 33
gtttttttttgtttt 23
t 37
tagtttttttttgtttt 21
tatatatattagtttttttttgtttt 12
tatatattagtttttttttgtttt 14
tatattagtttttttttgtttt 16
tattagtttttttttgtttt 18
tattatatatattagtttttttttgtttt 9
tcaaagattattatatatattagtttttttttgtttt 1
tgtttt 32
tt 36
ttagtttttttttgtttt 20
ttatatatattagtttttttttgtttt 11
ttattatatatattagtttttttttgtttt 8
ttgtttt 31
ttt 35
tttgtttt 30
tttt 34
ttttgtttt 29
tttttgtttt 28
ttttttgtttt 27
tttttttgtttt 26
ttttttttgtttt 25
tttttttttgtttt 24


In [959]:
string = "atcaaagattattatatatattagtttttttttgtttt"
Tp,alphabet,alphabet_inv = string_2_numbers(string,return_alphabet=True)
''.join(list(map(alphabet_inv.__getitem__,BWT(Tp))))

26 26


'caatttttttgtatatttaaatattaaatttgtttttg'

In [960]:
string = """
...
AAAAAAAAAAAAA
"""
string="".join(string.split("\n"))

In [961]:
len(string)

29903

In [970]:
Tp,alphabet,alphabet_inv = string_2_numbers(string[:30000],return_alphabet=True)

In [971]:
%timeit bwt = BWT(Tp)


8861 8861
13291 13291
19936 19936
8861 8861
13291 13291
19936 19936
8861 8861
13291 13291
19936 19936
8861 8861
13291 13291
19936 19936
8861 8861
13291 13291
19936 19936
8861 8861
13291 13291
19936 19936
8861 8861
13291 13291
19936 19936
8861 8861
13291 13291
19936 19936
873 ms ± 48.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
bwt = ''.join(list(map(alphabet_inv.__getitem__,bwt)))